<a href="https://colab.research.google.com/github/2DObjectDetector/T23/blob/main/Emma_Sloan_Ass2_T23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set-Up

In [ ]:
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/kitti3k_car_only
!cp -r /content/drive/MyDrive/kitti3k_car_only /content/kitti3k_car_only
dataset_yaml = "/content/kitti3k_car_only/kitti3k_car_only.yaml"

# install YOLO
!pip install -U ultralytics
from ultralytics import YOLO

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 7.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Training

In [ ]:
# fine tune on car only dataset using config 2 from t2.2

model_car_only = YOLO("yolo11n.pt")

car_only_results = model_car_only.train(
    data=dataset_yaml,
    epochs=40, # more epochs 40
    imgsz=640,
    batch=16,
    lr0=0.001,
    freeze=10,
    workers=2,
    cache="ram",
    device=0,
    exist_ok=True,
    pretrained=True,
)

Ultralytics 8.4.124 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/kitti3k_car_only/kitti3k_car_only.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nm

# Test Set Metrics

In [ ]:
# get metrics for test set
test_metrics_car_only = model_car_only.val(
    data=dataset_yaml,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    exist_ok=True
)

Ultralytics 8.4.124 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 130.4±14.3 MB/s, size: 827.6 KB)
val: Scanning /content/kitti3k_car_only/labels/test... 450 images, 56 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 210.3it/s 2.1s
val: New cache created: /content/kitti3k_car_only/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 4.8it/s 6.1s
                   all        450       1795      0.897      0.806      0.906      0.663
Speed: 0.8ms preprocess, 2.6ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/runs/detect/val


In [ ]:
print(test_metrics_car_only.names)
print(f"Car AP50: {test_metrics_car_only.box.map50:.3f}")
print(f"Car AP50-95: {test_metrics_car_only.box.map:.3f}")

{0: 'Car'}
Car AP50: 0.906
Car AP50-95: 0.663
